# Task 13: Direct Preference Optimization (DPO) and Pairwise Reward Alignment

## Objective

To align a language model using preference pairs containing chosen and rejected responses without using a traditional reinforcement learning loop.

## Technologies / Tools Used

- Python 3.10+
- PyTorch
- TRL
- Hugging Face Transformers
- Accelerate
- Google Colab

## Formula

### DPO Loss

L_DPO = -log σ(β[(log π(y_w|x) - log π_ref(y_w|x))
                 - (log π(y_l|x) - log π_ref(y_l|x))])

where:

y_w = chosen response

y_l = rejected response

π = policy model

π_ref = frozen reference model

## Step 1: Install and Import Libraries

Install TRL, Transformers, and Accelerate.

In [1]:
# Install required libraries

!pip -q install -U trl transformers accelerate datasets

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 10.9 MB/s eta 0:00:00


## Step 2: Load Policy and Reference Models

Use a small GPT-2 model for a fast educational demonstration. The reference model remains frozen.

In [2]:
# Use a lightweight model

model_name = "sshleifer/tiny-gpt2"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

tokenizer.pad_token = tokenizer.eos_token

# Policy model

policy_model = AutoModelForCausalLM.from_pretrained(
    model_name
)

# Reference model

reference_model = AutoModelForCausalLM.from_pretrained(
    model_name
)

reference_model.eval()

for param in reference_model.parameters():
    param.requires_grad = False

print("Policy and reference models loaded.")

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.51MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Policy and reference models loaded.


## Step 3: Create Preference Pairs

Create prompt-response pairs containing chosen and rejected responses.

In [4]:
# Preference dataset

preferences = [
    {
        "prompt": "Explain artificial intelligence.",
        "chosen": "Artificial intelligence enables machines to perform tasks that normally require human intelligence.",
        "rejected": "I do not know."
    },
    {
        "prompt": "What is Python?",
        "chosen": "Python is a high-level programming language.",
        "rejected": "Python is a type of hardware."
    }
]

for item in preferences:

    print("Prompt:", item["prompt"])
    print("Chosen:", item["chosen"])
    print("Rejected:", item["rejected"])
    print()

Prompt: Explain artificial intelligence.
Chosen: Artificial intelligence enables machines to perform tasks that normally require human intelligence.
Rejected: I do not know.

Prompt: What is Python?
Chosen: Python is a high-level programming language.
Rejected: Python is a type of hardware.



## Step 4: Calculate Pairwise Preference Loss

Calculate a simplified DPO-style preference loss using policy and reference scores.

In [5]:
# Example preference scores

policy_chosen = torch.tensor(2.4)
policy_rejected = torch.tensor(1.2)

reference_chosen = torch.tensor(2.0)
reference_rejected = torch.tensor(1.5)

beta = 0.1

policy_difference = (
    policy_chosen - policy_rejected
)

reference_difference = (
    reference_chosen - reference_rejected
)

loss = -torch.log(
    torch.sigmoid(
        beta *
        (policy_difference - reference_difference)
    )
)

print("Policy difference:", policy_difference.item())
print("Reference difference:", reference_difference.item())
print("DPO Loss:", loss.item())

Policy difference: 1.2000000476837158
Reference difference: 0.5
DPO Loss: 0.6587594747543335


## Conclusion

A simplified DPO preference-alignment pipeline was implemented using chosen and rejected responses. The policy model was compared against a frozen reference model using relative log-likelihood differences.